# 2026 MediaEval NewsImages




# The prompt


news image thumbnail, illustration, vector art, ___

# Method 1.1: CFT-CLIP

In [ ]:
# @title packages

!pip install -q diffusers transformers datasets accelerate safetensors sentencepiece open_clip_torch torch torchvision

!pip install easyocr matplotlib

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Get the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Log in
login(token=hf_token)

In [ ]:

# @title setup

import torch
import torch.nn.functional as F
import easyocr
from diffusers import AutoPipelineForText2Image
from transformers import AutoModel, AutoProcessor, CLIPProcessor, CLIPModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load SDXL-Turbo
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe.to(DEVICE)

# 2. Load OCR
reader = easyocr.Reader(['en'], gpu=(DEVICE == "cuda"))

# 3. Load Standard CLIP
standard_clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
standard_clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
standard_clip_model.eval()

# 4. Load CFT-CLIP
cft_clip_processor = AutoProcessor.from_pretrained("humane-lab/CFT-CLIP")
cft_clip_model = AutoModel.from_pretrained("humane-lab/CFT-CLIP").to(DEVICE)
cft_clip_model.eval()



In [ ]:

# @title model def

import re
import numpy as np
from PIL import Image, ImageOps

def sanitize_prompt(raw_text: str) -> str:
    text = re.sub(r'["\'].*?["\']', '', raw_text)
    text = re.sub(r'^.*?\-\s*', '', text)
    return text.strip()

def has_text(image: Image.Image, threshold: float = 0.5) -> bool:
    image_np = np.array(image)
    results = reader.readtext(image_np)
    for (bbox, text, prob) in results:
        if prob > threshold and len(text.strip()) > 1:
            return True
    return False

def score_image_with_standard_clip(image: Image.Image, text: str) -> float:
    inputs = standard_clip_processor(text=[text], images=image, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        outputs = standard_clip_model(**inputs)
    return outputs.logits_per_image.item()

def score_image_with_cft_clip(image: Image.Image, text: str) -> float:
    inputs = cft_clip_processor(text=[text], images=image, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        outputs = cft_clip_model(**inputs)
        text_embeds = F.normalize(outputs.text_embeds, p=2, dim=-1)
        image_embeds = F.normalize(outputs.image_embeds, p=2, dim=-1)
        similarity = torch.matmul(text_embeds, image_embeds.t())
    return similarity.item()

def generate_news_thumbnail(news_lead: str, max_attempts: int = 5, inference_steps: int = 1):
    clean_prompt = sanitize_prompt(news_lead)
    full_prompt = "news image thumbnail, illustration, vector art, " + clean_prompt

    best_image = None
    best_cft_score = -float('inf')
    fallback_image = None
    candidates_history = []

    for attempt in range(max_attempts):
        out = pipe(
            prompt=full_prompt,
            num_inference_steps=inference_steps,
            guidance_scale=0.0,
            height=512,
            width=512
        )
        candidate_image = out.images[0]
        candidate_image = ImageOps.fit(candidate_image, (460, 260), method=Image.Resampling.LANCZOS)

        if fallback_image is None:
            fallback_image = candidate_image

        is_rejected = has_text(candidate_image)
        std_score = -1.0
        cft_score = -1.0

        if not is_rejected:
            std_score = score_image_with_standard_clip(candidate_image, clean_prompt)
            cft_score = score_image_with_cft_clip(candidate_image, clean_prompt)

            if cft_score > best_cft_score:
                best_cft_score = cft_score
                best_image = candidate_image

        candidates_history.append({
            "image": candidate_image,
            "std_score": std_score,
            "cft_score": cft_score,
            "rejected": is_rejected
        })

    final_image = best_image if best_image is not None else fallback_image
    return final_image, candidates_history


In [ ]:

# @title eval

import matplotlib.pyplot as plt


MAX_ATTEMPT=20

# Test data
news_text = "The central bank decided to lower interest rates by 0.5% today."


# Execute pipeline
final_thumbnail, candidates = generate_news_thumbnail(news_text, max_attempts=MAX_ATTEMPT)

# Save final result
final_thumbnail.save("output_thumbnail.png", "PNG")

# Render grid layout
num_candidates = len(candidates)
cols = 3
rows = (num_candidates + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten()


for i, candidate in enumerate(candidates):
    ax = axes[i]
    ax.imshow(candidate["image"])
    ax.axis('off')

    title = f"Attempt {i+1}\n"
    if candidate["rejected"]:
        title += "REJECTED (Text Detected)"
        ax.set_title(title, color="red", fontsize=10, fontweight="bold")
    else:
        title += f"Std CLIP: {candidate['std_score']:.4f}\n"
        title += f"CFT-CLIP: {candidate['cft_score']:.4f}"
        if candidate["image"] == final_thumbnail:
            title += "\n[SELECTED]"
            ax.set_title(title, color="green", fontsize=10, fontweight="bold")
        else:
            ax.set_title(title, color="black", fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')




plt.tight_layout()
plt.show()



In [ ]:

min_cft = min(candidate["cft_score"] for candidate in candidates if not candidate["rejected"])
max_cft = max(candidate["cft_score"] for candidate in candidates )

print(f"Min CFT-CLIP Score: {min_cft}")
print(f"Max CFT-CLIP Score: {max_cft}")


min_clip = min(candidate["std_score"] for candidate in candidates if not candidate["rejected"])
max_clip = max(candidate["std_score"] for candidate in candidates )

print(f"Min CLIP Score: {min_clip}")
print(f"Max CLIP Score: {max_clip}")


display(final_thumbnail)


# 1.3 finetune

In [ ]:
from numba import cuda
# all of your code and execution
cuda.select_device(0)
cuda.close()

In [ ]:
!nvidia-smi

In [ ]:

# @title clean

gc.collect()               # Force Python garbage collection
torch.cuda.empty_cache()   # Force release PyTorch VRAM
# ==========================================


In [ ]:
# @title setup
!pip install --upgrade torchao
!pip install -q diffusers transformers peft accelerate datasets ftfy


import torch
import torch.nn.functional as F
from diffusers import AutoPipelineForText2Image, AutoencoderKL
from transformers import AutoProcessor, AutoModel
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
from google.colab import userdata
from huggingface_hub import login


In [ ]:
# @title load
import gc
import torch
from huggingface_hub import login
from google.colab import userdata

# ==========================================
# 🛑 OOM Prevention: Force release old models from memory
# ==========================================
print("Cleaning up old models from memory...")
for model_var in ['vae', 'pipeline', 'clip_model', 'clip_processor', 'optimizer']:
    if model_var in globals():
        del globals()[model_var]

gc.collect()               # Force Python garbage collection
torch.cuda.empty_cache()   # Force release PyTorch VRAM
# ==========================================

# Get the token from Colab Secrets
hf_token = userdata.get('HF_token')

# Log in
login(token=hf_token)

# Set device and precision
device = "cuda" if torch.cuda.is_available() else "cpu"
weight_dtype = torch.float16

# 1. Load the fixed VAE to prevent NaN errors during float16 decoding
print("Loading VAE...")
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=weight_dtype
).to(device)
vae.requires_grad_(False)

# 2. Load SDXL-Turbo base pipeline
print("Loading SDXL-Turbo...")
pipeline = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    vae=vae,
    torch_dtype=weight_dtype,
    variant="fp16"
).to(device)

pipeline.unet.requires_grad_(False)
pipeline.text_encoder.requires_grad_(False)
pipeline.text_encoder_2.requires_grad_(False)

# 3. Apply LoRA (Rank 8) to UNet
print("Applying LoRA...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
pipeline.unet = get_peft_model(pipeline.unet, lora_config)

# Enable gradient checkpointing to save VRAM on T4
pipeline.unet.enable_gradient_checkpointing()

# 4. Load Judge (CFT-CLIP)
print("Loading CFT-CLIP...")
clip_processor = AutoProcessor.from_pretrained("humane-lab/CFT-CLIP")
clip_model = AutoModel.from_pretrained(
    "humane-lab/CFT-CLIP",
    torch_dtype=weight_dtype
).to(device)
clip_model.requires_grad_(False)

print("Load complete!")

In [ ]:
# @title pipeline (cached embeddings)

from functools import lru_cache

CLIP_MEAN = torch.tensor([0.48145466, 0.4578275,  0.40821073], device=device).view(1,3,1,1)
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1,3,1,1)

pipeline.scheduler.set_timesteps(1, device=device)

# Cache text embeddings — both SDXL and CFT-CLIP text encoders
# so we never run them more than once per unique title
embedding_cache = {}

def get_embeddings_cached(prompt, article_title):
    if article_title not in embedding_cache:
        with torch.no_grad():
            # SDXL prompt embeddings
            prompt_embeds, _, pooled_prompt_embeds, _ = pipeline.encode_prompt(
                prompt=prompt, prompt_2=None, device=device,
                num_images_per_prompt=1, do_classifier_free_guidance=False
            )
            add_time_ids = torch.tensor(
                [[256, 256, 0, 0, 256, 256]],
                dtype=prompt_embeds.dtype, device=device
            )
            # CFT-CLIP text embedding
            text_inputs = clip_processor(
                text=[article_title], return_tensors="pt", padding=True
            ).to(device)
            dummy_px = torch.zeros(1, 3, 224, 224, device=device, dtype=weight_dtype)
            out = clip_model(pixel_values=dummy_px, **text_inputs)
            target_text_embeds = F.normalize(out.text_embeds.float(), dim=-1)

        embedding_cache[article_title] = (
            prompt_embeds, pooled_prompt_embeds, add_time_ids, target_text_embeds
        )

    return embedding_cache[article_title]


def differentiable_training_step(prompt, article_title, pipeline, clip_model, clip_processor, optimizer):
    pipeline.unet.train()
    optimizer.zero_grad()

    prompt_embeds, pooled_prompt_embeds, add_time_ids, target_text_embeds = \
        get_embeddings_cached(prompt, article_title)

    added_cond_kwargs = {"text_embeds": pooled_prompt_embeds, "time_ids": add_time_ids}

    latents = torch.randn((1, 4, 32, 32), device=device, dtype=weight_dtype)
    timesteps = torch.tensor([999], device=device)

    with torch.amp.autocast('cuda', dtype=weight_dtype):
        scaled_latents = pipeline.scheduler.scale_model_input(latents, timesteps[0])
        noise_pred = pipeline.unet(
            scaled_latents, timesteps,
            encoder_hidden_states=prompt_embeds,
            added_cond_kwargs=added_cond_kwargs,
        ).sample

        pipeline.scheduler._step_index = 0
        x0 = pipeline.scheduler.step(noise_pred, timesteps[0], latents).pred_original_sample
        decoded_image = pipeline.vae.decode(x0 / pipeline.vae.config.scaling_factor).sample

    decoded_image = decoded_image.float()
    decoded_image = (decoded_image / 2 + 0.5).clamp(0, 1)
    clip_input_image = F.interpolate(
        decoded_image, size=(224, 224),
        mode="bilinear", align_corners=False, antialias=True
    )
    clip_input_image = (clip_input_image - CLIP_MEAN) / CLIP_STD

    # Only run the vision encoder — text target already cached
    image_out = clip_model.vision_model(pixel_values=clip_input_image.to(weight_dtype))
    image_proj = clip_model.visual_projection(image_out.pooler_output)
    image_embeds = F.normalize(image_proj.float(), dim=-1)

    similarity_score = (image_embeds * target_text_embeds).sum(dim=-1).mean()


    # TODO
    # loss = 1.0 - similarity_score
    EPS = 1e-6
    loss = -torch.log((similarity_score + 1) / 2 + EPS)

    loss.backward()
    optimizer.step()

    return loss.item(), similarity_score.item()

In [ ]:
# @title Load training data
import pandas as pd
import re
import ftfy
from torch.utils.data import Dataset, DataLoader

df = pd.read_csv("news_articles.csv")
print(f"Total articles before cleaning: {len(df)}")

def clean_title(title: str) -> str:
    title = ftfy.fix_text(title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title

class NewsThumbnailDataset(Dataset):
    def __init__(self, dataframe, prompt_prefix="news image thumbnail, illustration, vector art, "):
        self.prefix = prompt_prefix
        titles = dataframe["article_title"].dropna().tolist()
        cleaned = [clean_title(t) for t in titles]
        tokens = clip_processor(
            text=cleaned, return_tensors="pt",
            padding=True, truncation=False
        )
        valid = [
            t for t, ids in zip(cleaned, tokens["input_ids"])
            if ids.ne(0).sum().item() <= 77
        ]
        print(f"Kept {len(valid)}/{len(titles)} titles after cleaning")
        self.titles = valid

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        title = self.titles[idx]
        return {
            "prompt": self.prefix + title,
            "article_title": title
        }

dataset = NewsThumbnailDataset(df)

val_size   = min(50, int(0.05 * len(dataset)))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)

print(f"Train: {train_size} | Val: {val_size}")

In [ ]:
# @title Full fine-tuning run
import os, gc
import matplotlib.pyplot as plt

# ── Config ──────────────────────────────────────────────
SAVE_PATH     = "/content/drive/MyDrive/cft_optimized_model"
LR            = 1e-5
EPOCHS        = 3   # 2 or 3
MAX_STEPS     = 999999  # 9999999 INF
LOG_EVERY     = 100
VAL_EVERY     = 500
VAL_SAMPLES   = 10
CKPT_EVERY    = 4000 # 4000
# ────────────────────────────────────────────────────────
os.makedirs(SAVE_PATH, exist_ok=True)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, pipeline.unet.parameters()),
    lr=LR
)

history = {"step": [], "loss": [], "score": [], "val_score": []}
global_step = 0

def run_val(n=VAL_SAMPLES):
    """Score n validation items with the frozen inference pipeline."""
    pipeline.unet.eval()
    scores = []
    val_items = list(val_loader)[:n]
    with torch.no_grad():
        for batch in val_items:
            title = batch["article_title"][0]
            img = pipeline(
                batch["prompt"][0],
                num_inference_steps=1,
                guidance_scale=0.0,
                height=512, width=512
            ).images[0]
            # reuse clip_processor + clip_model for scoring
            from PIL import Image
            inputs = clip_processor(
                text=[title], images=img,
                return_tensors="pt", padding=True
            ).to(device)
            out = clip_model(**inputs)
            t = F.normalize(out.text_embeds.float(), dim=-1)
            i = F.normalize(out.image_embeds.float(), dim=-1)
            scores.append((t * i).sum().item())
    pipeline.unet.train()
    return sum(scores) / len(scores)

print("Starting full fine-tuning...")
print(f"  Epochs: {EPOCHS}  |  Steps/epoch: {len(train_loader)}  |  LR: {LR}\n")

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    epoch_score = 0.0
    n_ok = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        prompt        = batch["prompt"][0]
        article_title = batch["article_title"][0]

        try:
            loss, score = differentiable_training_step(
                prompt, article_title,
                pipeline, clip_model, clip_processor, optimizer
            )
            epoch_loss  += loss
            epoch_score += score
            n_ok += 1
            global_step += 1

            # [TODO] early stop
            if global_step >= MAX_STEPS:
                print(f"Reached MAX_STEPS ({MAX_STEPS}), stopping early.")
                break

            history["step"].append(global_step)
            history["loss"].append(loss)
            history["score"].append(score)

            if global_step % LOG_EVERY == 0:
                print(f"  step {global_step} | loss {loss:.4f} | score {score:.4f}")

            if global_step % VAL_EVERY == 0:
                val_score = run_val()
                history["val_score"].append((global_step, val_score))
                print(f"  ── val CFT-CLIP score: {val_score:.4f} ──")

            if global_step % CKPT_EVERY == 0:
                ckpt_dir = os.path.join(SAVE_PATH, f"step_{global_step}")
                pipeline.unet.save_pretrained(ckpt_dir)
                print(f"  checkpoint saved → {ckpt_dir}")

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"  OOM at step {global_step}, skipping")
                torch.cuda.empty_cache()
                gc.collect()
            else:
                raise

    if n_ok > 0:
        print(f"\nEpoch {epoch+1} done | "
              f"avg loss {epoch_loss/n_ok:.4f} | "
              f"avg score {epoch_score/n_ok:.4f}\n")

# ── Final save ───────────────────────────────────────────
pipeline.unet.save_pretrained(SAVE_PATH)
print(f"Final LoRA weights saved → {SAVE_PATH}")

# ── Loss curve ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history["step"], history["loss"], alpha=0.4, linewidth=0.8, label="per-step loss")
window = 50
if len(history["loss"]) >= window:
    smoothed = pd.Series(history["loss"]).rolling(window).mean()
    ax1.plot(history["step"], smoothed, linewidth=1.5, label=f"rolling mean ({window})")
ax1.set_xlabel("step"); ax1.set_ylabel("loss"); ax1.legend()
ax1.set_title("Training loss (1 - CFT-CLIP score)")

ax2.plot(history["step"], history["score"], alpha=0.4, linewidth=0.8, color="green")
if len(history["score"]) >= window:
    smoothed_s = pd.Series(history["score"]).rolling(window).mean()
    ax2.plot(history["step"], smoothed_s, linewidth=1.5, color="darkgreen")
if history["val_score"]:
    vx, vy = zip(*history["val_score"])
    ax2.scatter(vx, vy, color="red", zorder=5, label="val score")
    ax2.legend()
ax2.set_xlabel("step"); ax2.set_ylabel("CFT-CLIP score"); ax2.set_title("Reward score")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "training_curve.png"), dpi=150)
plt.show()

In [ ]:
# @title Eval — finetuned vs baseline, 20 val articles, best-of-10

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import random

N_ARTICLES   = 5
CANDIDATES   = 10
PROMPT_PREFIX = "news image thumbnail, illustration, vector art, "

# ── Temporarily detach LoRA to get the frozen baseline ───────────────────────
def best_of_n(article_title, prompt, n=CANDIDATES):
    best_img, best_score = None, -float("inf")
    for _ in range(n):
        with torch.no_grad():
            img = pipeline(
                prompt,
                num_inference_steps=1,
                guidance_scale=0.0,
                height=512, width=512
            ).images[0]
            inputs = clip_processor(
                text=[article_title], images=img,
                return_tensors="pt", padding=True
            ).to(device)
            out = clip_model(**inputs)
            t = F.normalize(out.text_embeds.float(), dim=-1)
            i = F.normalize(out.image_embeds.float(), dim=-1)
            score = (t * i).sum().item()
        if score > best_score:
            best_score = score
            best_img = img
    return best_img, best_score

# Sample 20 val articles
eval_ds = NewsThumbnailDataset(
    df, prompt_prefix=PROMPT_PREFIX
)
_, val_proper = torch.utils.data.random_split(
    eval_ds,
    [len(eval_ds) - min(50, int(0.05 * len(eval_ds))),
      min(50, int(0.05 * len(eval_ds)))],
    generator=torch.Generator().manual_seed(42)
)
indices = random.sample(range(len(val_proper)), N_ARTICLES)
eval_items = [val_proper[i] for i in indices]

results = []
for item in tqdm(eval_items, desc="Tuned model"):
    title  = item["article_title"]
    prompt = item["prompt"]
    pipeline.unet.eval()
    img_tuned, score_tuned = best_of_n(title, prompt)

    # Disable LoRA adapters → frozen base model
    pipeline.unet.disable_adapter_layers()
    img_base, score_base = best_of_n(title, prompt)
    pipeline.unet.enable_adapter_layers()

    results.append({
        "title":       title,
        "img_base":    img_base,
        "score_base":  score_base,
        "img_tuned":   img_tuned,
        "score_tuned": score_tuned,
        "delta":       score_tuned - score_base,
    })

# ── Display ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, N_ARTICLES * 3.2))
gs  = gridspec.GridSpec(N_ARTICLES, 3, width_ratios=[1, 1, 2], hspace=0.5, wspace=0.05)

for idx, r in enumerate(results):
    # Baseline image
    ax_base = fig.add_subplot(gs[idx, 0])
    ax_base.imshow(r["img_base"])
    ax_base.axis("off")
    ax_base.set_title(f"Base: {r['score_base']:.4f}", fontsize=9, color="gray")

    # Tuned image
    ax_tuned = fig.add_subplot(gs[idx, 1])
    ax_tuned.imshow(r["img_tuned"])
    ax_tuned.axis("off")
    delta_color = "green" if r["delta"] > 0 else "red"
    ax_tuned.set_title(
        f"Tuned: {r['score_tuned']:.4f}  (Δ {r['delta']:+.4f})",
        fontsize=9, color=delta_color
    )

    # Article title
    ax_txt = fig.add_subplot(gs[idx, 2])
    ax_txt.axis("off")
    ax_txt.text(0.02, 0.5, r["title"], fontsize=10,
                va="center", wrap=True, transform=ax_txt.transAxes)

plt.suptitle(
    f"Base vs LoRA-tuned — best-of-{CANDIDATES} per article",
    fontsize=13, y=1.001
)
plt.savefig(os.path.join(SAVE_PATH, "eval_comparison.png"), dpi=120, bbox_inches="tight")
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────────
deltas = [r["delta"] for r in results]
wins   = sum(1 for d in deltas if d > 0)
print(f"\nAvg CFT-CLIP  base:  {sum(r['score_base']  for r in results)/N_ARTICLES:.4f}")
print(f"Avg CFT-CLIP tuned:  {sum(r['score_tuned'] for r in results)/N_ARTICLES:.4f}")
print(f"Avg delta:           {sum(deltas)/N_ARTICLES:+.4f}")
print(f"Tuned wins:          {wins}/{N_ARTICLES}")

In [ ]:

# @title Hail Mary — test set eval, 20 articles, best-of-10

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import re, ftfy, random

PROMPT_PREFIX = "news image thumbnail, illustration, vector art, "
N_ARTICLES    = 20
CANDIDATES    = 10

# Load and clean test set
df_test = pd.read_csv("news_articles_test.csv", encoding="latin1")

def clean_title(title: str) -> str:
    title = ftfy.fix_text(title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title

raw_titles = df_test["article_title"].dropna().tolist()
cleaned    = [clean_title(t) for t in raw_titles]

# Drop titles over 77 tokens
tokens = clip_processor(
    text=cleaned, return_tensors="pt",
    padding=True, truncation=False
)
valid_titles = [
    t for t, ids in zip(cleaned, tokens["input_ids"])
    if ids.ne(0).sum().item() <= 77
]
print(f"Test set: kept {len(valid_titles)}/{len(raw_titles)} after cleaning")

sampled = random.sample(valid_titles, N_ARTICLES)

def best_of_n(article_title, prompt, n=CANDIDATES):
    best_img, best_score = None, -float("inf")
    for _ in range(n):
        with torch.no_grad():
            img = pipeline(
                prompt,
                num_inference_steps=1,
                guidance_scale=0.0,
                height=256, width=256
            ).images[0]
            inputs = clip_processor(
                text=[article_title], images=img,
                return_tensors="pt", padding=True
            ).to(device)
            out = clip_model(**inputs)
            t = F.normalize(out.text_embeds.float(), dim=-1)
            i = F.normalize(out.image_embeds.float(), dim=-1)
            score = (t * i).sum().item()
        if score > best_score:
            best_score = score
            best_img = img
    return best_img, best_score

results = []
for title in tqdm(sampled, desc="Test eval"):
    prompt = PROMPT_PREFIX + title

    pipeline.unet.eval()
    img_tuned, score_tuned = best_of_n(title, prompt)

    pipeline.unet.disable_adapter_layers()
    img_base, score_base = best_of_n(title, prompt)
    pipeline.unet.enable_adapter_layers()

    results.append({
        "title":       title,
        "img_base":    img_base,
        "score_base":  score_base,
        "img_tuned":   img_tuned,
        "score_tuned": score_tuned,
        "delta":       score_tuned - score_base,
    })

# Display
fig = plt.figure(figsize=(18, N_ARTICLES * 3.2))
gs  = gridspec.GridSpec(N_ARTICLES, 3, width_ratios=[1, 1, 2], hspace=0.5, wspace=0.05)

for idx, r in enumerate(results):
    ax_base = fig.add_subplot(gs[idx, 0])
    ax_base.imshow(r["img_base"])
    ax_base.axis("off")
    ax_base.set_title(f"Base: {r['score_base']:.4f}", fontsize=9, color="gray")

    ax_tuned = fig.add_subplot(gs[idx, 1])
    ax_tuned.imshow(r["img_tuned"])
    ax_tuned.axis("off")
    ax_tuned.set_title(
        f"Tuned: {r['score_tuned']:.4f}  (Δ {r['delta']:+.4f})",
        fontsize=9, color="green" if r["delta"] > 0 else "red"
    )

    ax_txt = fig.add_subplot(gs[idx, 2])
    ax_txt.axis("off")
    ax_txt.text(0.02, 0.5, r["title"], fontsize=10,
                va="center", wrap=True, transform=ax_txt.transAxes)

plt.suptitle(
    f"Test set — Base vs LoRA-tuned — best-of-{CANDIDATES} per article",
    fontsize=13, y=1.001
)
plt.savefig(os.path.join(SAVE_PATH, "test_eval.png"), dpi=120, bbox_inches="tight")
plt.show()

deltas = [r["delta"] for r in results]
wins   = sum(1 for d in deltas if d > 0)
print(f"\nAvg base score:  {sum(r['score_base']  for r in results)/N_ARTICLES:.4f}")
print(f"Avg tuned score: {sum(r['score_tuned'] for r in results)/N_ARTICLES:.4f}")
print(f"Avg delta:       {sum(deltas)/N_ARTICLES:+.4f}")
print(f"Tuned wins:      {wins}/{N_ARTICLES}")

# 1.4 SOTA models

In [ ]:
# @title setup

!pip install -q diffusers transformers accelerate peft easyocr bitsandbytes torch torchvision matplotlib numpy


from google.colab import userdata
from huggingface_hub import login

# Get the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Log in
login(token=hf_token)


PROMPT_PREFIX="news image thumbnail, illustration, vector art, "


In [ ]:
import gc
import torch


# @title Clear CUDA Memory
torch.cuda.empty_cache()
gc.collect()


!nvidia-smi


In [ ]:

# @title playground flux-klein 4B
import torch
from diffusers import DiffusionPipeline
from diffusers.quantizers import PipelineQuantizationConfig

quant_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": torch.bfloat16,
    },
    components_to_quantize=["transformer", "text_encoder"],
)

DIFFUSION_MODEL = "black-forest-labs/FLUX.2-klein-4B"

pipe = DiffusionPipeline.from_pretrained(
    DIFFUSION_MODEL,
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,   # load weights to CPU before moving to GPU
)

# DO NOT call pipe.to(device) — use offload instead
pipe.enable_model_cpu_offload()   # moves layers to GPU only during forward pass




In [ ]:

# @title image gen



news_text = "The central bank decided to lower interest rates by 0.5% today."

prompt = PROMPT_PREFIX + news_text

image = pipe(
    prompt=prompt,
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=8,
    generator=torch.Generator("cpu").manual_seed(10),  # use CPU generator with offload
).images[0]

image.save("flux-klein.png")
display(image)
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")



# 1.5 GGUF models


In [ ]:
# @title setup

!pip install -q diffusers transformers accelerate peft easyocr bitsandbytes torch torchvision matplotlib numpy


from google.colab import userdata
from huggingface_hub import login

# Get the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Log in
login(token=hf_token)


PROMPT_PREFIX="news image thumbnail, illustration"



In [ ]:

# @title playground ZImageTurbo
import torch
from diffusers import DiffusionPipeline
from diffusers.quantizers import PipelineQuantizationConfig

quant_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": torch.bfloat16,
    },
    components_to_quantize=["transformer", "text_encoder"],
)

DIFFUSION_MODEL = "Tongyi-MAI/Z-Image-Turbo"

pipe = DiffusionPipeline.from_pretrained(
    DIFFUSION_MODEL,
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,   # load weights to CPU before moving to GPU
)

# DO NOT call pipe.to(device) — use offload instead
pipe.enable_model_cpu_offload()   # moves layers to GPU only during forward pass




In [ ]:

# @title image gen

PROMPT_PREFIX="news image thumbnail, illustration"

news_text = "The central bank decided to lower interest rates by 0.5% today."

prompt = PROMPT_PREFIX + news_text

image = pipe(
    prompt=prompt,
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=8,
    generator=torch.Generator("cpu").manual_seed(10),  # use CPU generator with offload
).images[0]

image.save("flux-klein.png")
display(image)
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

